In [8]:
import numpy as np
import os
import matplotlib.pyplot as plt

# %matplotlib inline
import rasterio
import pyproj
from rasterio.warp import calculate_default_transform, reproject, Resampling
import os
from rasterio.plot import show

# Allow inline jshtml animations
from matplotlib import rc
rc('animation', html='jshtml')
import anuga

import anuga
from anuga import Reflective_boundary, Dirichlet_boundary
from netCDF4 import Dataset
import numpy as np
from netCDF4 import Dataset
import matplotlib.pyplot as plt
import matplotlib.tri as tri
import numpy as np


# Load dem data

In [33]:
data_dir = '/storage/group/cxs1024/default/mehdi/Hurricane_MatthewData/DEM10/'
topography_file = os.path.join(data_dir,'DM10GLDN2.asc')


with rasterio.open(topography_file) as src:
    # Get bounds
    x_min, y_min, x_max, y_max = src.bounds
    
    print(f"X range: {x_min} to {x_max}")
    print(f"Y range: {y_min} to {y_max}")
    
# Create bounding polygon (counter-clockwise from bottom left)
bounding_polygon = [[x_min, y_min],  # Bottom left
                   [x_min, y_max],   # Top left
                   [x_max, y_max],   # Top right
                   [x_max, y_min]]   # Bottom right

# # Polygon defining broad area of interest
finer_zone = anuga.read_polygon('finer_zone.csv')

# Resolution for most of the mesh
base_resolution = 50_000.0  # m^2

# Resolution in particular area of interest 
finer_zone_resolution = 25_000.0 # m^2

interior_regions = [[finer_zone, finer_zone_resolution]]

X range: 201527.93407489875 to 248417.93407489875
Y range: 3898940.192838932 to 3922540.192838932


# Loading State States

In [34]:
# === 1. Hot-start from existing SWW file ===
sww_file = 'results/Hurricane_steady_state_final.sww'  # Change if needed

nc = Dataset(sww_file, 'r')

# Extract mesh
x = nc.variables['x'][:]
y = nc.variables['y'][:]
volumes = nc.variables['volumes'][:]  # triangle connectivity


# Load final state (use centroid values with _c suffix)
last_index = -1
stage = nc.variables['stage_c'][last_index, :]
xmomentum = nc.variables['xmomentum_c'][last_index, :]
ymomentum = nc.variables['ymomentum_c'][last_index, :]
elevation = nc.variables['elevation_c'][:]

# Optional: friction at centroids
if 'friction_c' in nc.variables:
    friction = nc.variables['friction_c'][:]


# Create domain

In [ ]:
domain = anuga.create_domain_from_regions(
    bounding_polygon=bounding_polygon,

    boundary_tags={'west':  [0],
                   'north': [1],
                   'east':  [2],
                   'south': [3]},               
    
    maximum_triangle_area=base_resolution,
    interior_regions=interior_regions,
    mesh_filename='mesh/Hurricane_flood_inundation_simulation.msh')

# SET OUTPUT FILE NAME
domain.set_name('results/Hurricane_flood_inundation_simulation')

/storage/work/amb10399/envs/MHPI_FLOOD/lib/python3.9/site-packages/anuga/structures/riverwall.py:124: RuntimeWarning: invalid value encountered in cast
  self.hydraulic_properties_rowIndex=numpy.array([default_int]).astype(int)


# Set quantities

In [ ]:
domain.set_quantity('elevation', elevation, location='centroids', verbose=True)
domain.set_quantity('stage', stage, location='centroids', verbose=True)

domain.set_quantity('xmomentum', xmomentum, location='centroids', verbose=True)
domain.set_quantity('ymomentum', ymomentum, location='centroids', verbose=True)

if 'friction_c' in nc.variables:
    domain.set_quantity('friction', friction, location='centroids', verbose=True)

nc.close()

# Set Boundary conditions

In [ ]:
low_stage = -10
Br = anuga.Reflective_boundary(domain)
Bt = anuga.Transmissive_boundary(domain)

outflow_BC = anuga.Dirichlet_boundary([low_stage, 0.0, 0.0]) 

domain.set_boundary({'south':   Br, # wall
                     'east':    outflow_BC, # outflow
                     'north':   outflow_BC, # outflow
                     'west':    Br})


# Enforcing Stream flow histories on gauges!


In [ ]:
# ================= CONFIGURATION =================
TMS_OUTPUT_FOLDER = '/storage/group/cxs1024/default/mehdi/Hurricane_MatthewData/tms_files'
radius = 120.0  
max_final_time = 0.0  # Initialize to store the longest duration found

gauges = [
    {"id": "Bc1_8791413",  "x": 203142.27, "y": 3922226.29},
    {"id": "Bc2_8790751",  "x": 209767.46, "y": 3915832.46},
    {"id": "Bc3_8790719",  "x": 214258.80, "y": 3920627.66},
    {"id": "Bc4_8790801",  "x": 215750.40, "y": 3914499.44},
    {"id": "Bc5_8790519",  "x": 218575.49, "y": 3920699.28},
    {"id": "Bc6_8790559",  "x": 225078.15, "y": 3921002.98},
    {"id": "Bc7_11235707", "x": 227582.00, "y": 3915030.09},
    {"id": "Bc8_11236643", "x": 231177.77, "y": 3904981.34},
]

print("Setting up Time-Series Inflows and calculating Max Duration...")

for gauge in gauges:
    g_id = gauge['id']
    center = (gauge['x'], gauge['y'])
    usgs_id = g_id.split('_')[-1]
    tms_filename = os.path.join(TMS_OUTPUT_FOLDER, f"{usgs_id}.tms")
    
    if os.path.exists(tms_filename):
        # 1. Update max_final_time by reading the NetCDF file
        with Dataset(tms_filename, 'r') as nc:
            file_end_time = nc.variables['time'][-1]
            if file_end_time > max_final_time:
                max_final_time = file_end_time
        
        # 2. Setup the ANUGA Operator
        Q_function = anuga.file_function(tms_filename, quantities='discharge')
        region = anuga.Region(domain, center=center, radius=radius)
        anuga.Inlet_operator(domain, region, Q=Q_function)
        
        print(f"  -> Added Inlet at {g_id}. File ends at {file_end_time/3600:.1f} hrs")
    else:
        print(f"  !! Warning: TMS file not found for {g_id}")


# Convert from MaskedArray to standard float
if hasattr(max_final_time, 'item'):
    final_time_seconds = float(max_final_time.item())
else:
    final_time_seconds = float(max_final_time)


print(f"\nConfiguration Complete.")
print(f"Global Simulation Final Time set to: {final_time_seconds} seconds ({final_time_seconds/3600:.2f} hours)")

# Plot computational domain and topography

In [ ]:
plt.figure(figsize=(14, 10))

nodes = domain.get_nodes(absolute=True) 
triangles = domain.get_triangles()
elev_nodes = domain.get_quantity('elevation').get_values(location='unique vertices') # [cite: 1895]

triangulation = tri.Triangulation(nodes[:, 0], nodes[:, 1], triangles)

tpc = plt.tripcolor(triangulation, elev_nodes, cmap='terrain', shading='flat', alpha=0.8)
cbar = plt.colorbar(tpc)
cbar.set_label('Elevation (m)')

plt.triplot(triangulation, color='black', linewidth=0.1, alpha=0.3)

gauge_x = [g['x'] for g in gauges]
gauge_y = [g['y'] for g in gauges]
gauge_ids = [g['id'] for g in gauges]

plt.scatter(gauge_x, gauge_y, color='red', s=60, edgecolors='white', marker='^', zorder=10, label='Inflow Gauges')

for i, txt in enumerate(gauge_ids):
    plt.annotate(txt, (gauge_x[i], gauge_y[i]), xytext=(5, 5), 
                 textcoords='offset points', fontsize=9, fontweight='bold', color='black')

plt.title("Watershed Topography and Gauge Locations")
plt.xlabel("Easting (m)")
plt.ylabel("Northing (m)")
plt.gca().set_aspect('equal')

plt.legend(loc='upper right') 

plt.tight_layout()
plt.show()

In [ ]:
minimum_allowed_height = 0.025
minimum_storable_height = 0.025

domain.set_minimum_allowed_height(minimum_allowed_height)
domain.set_minimum_storable_height(minimum_storable_height)

dplotter = anuga.Domain_plotter(domain)


# EVOLVE UNTIL END

In [ ]:
DAY = 24 * 3600

for t in domain.evolve(yieldstep=0.125 * DAY, finaltime=final_time_seconds):
    dplotter.save_depth_frame(vmin=0.05, vmax=5)
    domain.print_timestepping_statistics()
    print(f"Progress: {100 * t / final_time_seconds:.1f}%")

In [ ]:
dplotter.make_depth_animation()